# Stage 18 V1 — FP/FN и operating modes текущего baseline

## Исследовательский вопрос

**Как меняется баланс ложноположительных и ложноотрицательных решений
текущего принятого baseline при разных operating modes
и можно ли на этой основе обоснованно выделить кандидата
на зону дополнительной проверки?**

---

## Зачем этот вопрос сейчас

Stages 1–17 уже показали, что:

- воспроизводимый baseline без `Q_B1_norm` / `Q_B2_norm` существует;
- дальнейший model-only search на неизменных 47 признаках имеет низкий ожидаемый information gain;
- существует common blind spot из 805 дефолтов;
- historical enrichment текущего датасета заблокирован из-за отсутствия восстанавливаемого temporal anchor.

Следующий практический вопрос заказчика относится уже не к очередной архитектуре модели,
а к **ошибкам принятия решения**:

- сколько дефолтов остаётся пропущено;
- сколько недефолтов получает ложный риск-сигнал;
- как меняется этот обмен при разных режимах работы;
- где может потребоваться дополнительная ручная или бизнес-проверка.

---

## Что остаётся неизменным

В Stage 18 не меняются:

- dataset `Data_final.xlsb`;
- target `DefMark`;
- identifier `INN`;
- working sample;
- сохранённый CV/OOF protocol;
- 47 разрешённых признаков;
- исключение `Q_B1_norm` и `Q_B2_norm` из predictors;
- accepted baseline / comparator;
- сохранённые OOF predictions.

Новые модели не обучаются.

Новые признаки не создаются.

Final test не используется.

Threshold не подбирается по final test и не объявляется «оптимальным».

---

## Что исследуется

Stage 18 рассматривает threshold как **business operating parameter**, а не как новый ML-hyperparameter.

Для нескольких заранее зафиксированных operating modes будут рассчитаны:

- Recall;
- Precision;
- F1;
- False Negative count;
- False Positive count;
- False Negative Rate;
- False Positive Rate;
- доля выборки с положительным risk decision.

Отдельно будет проверено, как разные operating modes относятся к ранее найденной группе
из **805 common blind-spot defaults**.

---

## Важное ограничение

Без подтверждённой стоимости FN и FP нельзя математически определить
единственный «лучший» threshold.

Поэтому Stage 18 не ищет универсальный оптимум.

Его задача — построить **карту компромиссов**,
которую затем можно связать с бизнес-процессом Комуса:

`автоматическое решение → дополнительная проверка → обычный поток`.

Бизнес-ориентир Recall около **69%** рассматривается только как reference point,
а не как обязательная цель оптимизации.

---

## Критерий завершения Stage 18

Stage считается завершённым, если получены:

1. воспроизводимая FP/FN-карта accepted OOF baseline;
2. сравнение нескольких заранее определённых operating modes;
3. положение 805 common blind-spot defaults относительно этих режимов;
4. кандидат на зону дополнительной проверки либо evidence,
   что выделять её по текущему score нерационально;
5. чёткое разделение:
   **FACTS / INTERPRETATION / LIMITATIONS / NEXT STEP**.

Следующий этап может обсуждать бизнес-правила или новые признаки
только после получения этой карты.

In [4]:
# ============================================================
# 1.1 Загрузка принятого OOF evidence
#
# Проверяем:
# - используем сохранённые Stage 3 OOF predictions;
# - working sample содержит ожидаемые 289 614 строк;
# - target и три baseline-модели имеют одинаковую длину;
# - accepted comparator GBDT_mean восстанавливается как среднее
#   CatBoost / XGBoost / LightGBM;
# - final test в Stage 18 не используется.
#
# Нового обучения здесь нет.
# ============================================================

from pathlib import Path
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


started = time.monotonic()

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OOF_PATH = (
    ROOT
    / "reports"
    / "generated"
    / "stage3_oof_predictions_V1.npz"
)

assert OOF_PATH.exists(), f"Не найден OOF artifact: {OOF_PATH}"

print("▶ Загружаю сохранённый OOF evidence...", flush=True)

with np.load(OOF_PATH) as data:
    available_keys = list(data.files)

    required_keys = [
        "working_indices",
        "target",
        "oof_catboost",
        "oof_xgboost",
        "oof_lightgbm",
        "consensus_rank",
        "rank_spread",
    ]

    missing_keys = [
        key for key in required_keys
        if key not in available_keys
    ]

    assert not missing_keys, (
        f"В OOF artifact отсутствуют ключи: {missing_keys}"
    )

    working_indices = data["working_indices"].copy()
    y_oof = data["target"].astype(int).copy()

    oof_catboost = data["oof_catboost"].astype(float).copy()
    oof_xgboost = data["oof_xgboost"].astype(float).copy()
    oof_lightgbm = data["oof_lightgbm"].astype(float).copy()

    consensus_rank = data["consensus_rank"].astype(float).copy()
    rank_spread = data["rank_spread"].astype(float).copy()


n_rows = len(y_oof)

assert n_rows == 289_614, (
    f"Неожиданный размер working sample: {n_rows}"
)

for name, values in {
    "working_indices": working_indices,
    "oof_catboost": oof_catboost,
    "oof_xgboost": oof_xgboost,
    "oof_lightgbm": oof_lightgbm,
    "consensus_rank": consensus_rank,
    "rank_spread": rank_spread,
}.items():
    assert len(values) == n_rows, (
        f"{name}: длина {len(values)} != {n_rows}"
    )

assert set(np.unique(y_oof)).issubset({0, 1})

# Accepted Stage 7 comparator:
# простое среднее трёх сохранённых baseline OOF probabilities.
gbdt_mean_oof = np.mean(
    np.column_stack(
        [
            oof_catboost,
            oof_xgboost,
            oof_lightgbm,
        ]
    ),
    axis=1,
)

auc = roc_auc_score(y_oof, gbdt_mean_oof)
gini = 2 * auc - 1

EXPECTED_GINI = 0.8063993952

GINI_TOLERANCE = 1e-8
gini_delta = abs(gini - EXPECTED_GINI)

assert gini_delta < GINI_TOLERANCE, (
    f"GBDT_mean identity check failed: "
    f"{gini:.10f} != {EXPECTED_GINI:.10f}; "
    f"delta={gini_delta:.2e}"
)

elapsed = time.monotonic() - started


print("✅ OOF evidence загружен")
print()
print("Контроль:")
print("working rows:", n_rows)
print("defaults:", int(y_oof.sum()))
print("non-defaults:", int((y_oof == 0).sum()))
print("GBDT_mean OOF Gini:", f"{gini:.10f}")
print("expected Gini:", f"{EXPECTED_GINI:.10f}")
print("final test used:", False)
print("model training:", False)
print("elapsed:", f"{elapsed:.2f} сек.")

▶ Загружаю сохранённый OOF evidence...
✅ OOF evidence загружен

Контроль:
working rows: 289614
defaults: 28015
non-defaults: 261599
GBDT_mean OOF Gini: 0.8063993934
expected Gini: 0.8063993952
final test used: False
model training: False
elapsed: 0.20 сек.


In [5]:
# ============================================================
# 2. Фиксация operating modes до просмотра FP/FN
#
# Единственное изменение между режимами:
# доля working sample, получающая положительный risk decision.
#
# Метрики здесь НЕ вычисляются.
# ============================================================

OPERATING_MODES = {
    "Narrow": 0.10,
    "Moderate": 0.15,
    "Balanced": 0.20,
    "Broad": 0.25,
    "High-recall": 0.30,
}


mode_rows = []

for mode_name, capacity in OPERATING_MODES.items():
    threshold = float(
        np.quantile(
            gbdt_mean_oof,
            1.0 - capacity,
        )
    )

    positive_decisions = int(
        np.sum(gbdt_mean_oof >= threshold)
    )

    actual_capacity = (
        positive_decisions / len(gbdt_mean_oof)
    )

    mode_rows.append(
        {
            "Режим": mode_name,
            "Плановая risk capacity": capacity,
            "Threshold": threshold,
            "Фактическая доля risk decision": actual_capacity,
            "Количество risk decision": positive_decisions,
        }
    )


operating_modes_table = pd.DataFrame(mode_rows)


print("Зафиксированные operating modes")
print("--------------------------------")

display(
    operating_modes_table.style.format(
        {
            "Плановая risk capacity": "{:.1%}",
            "Threshold": "{:.6f}",
            "Фактическая доля risk decision": "{:.2%}",
        }
    )
)

print()
print("Количество режимов:", len(operating_modes_table))
print("Метрики FP/FN рассчитаны:", False)
print("Final test использован:", False)

Зафиксированные operating modes
--------------------------------


,Режим,Плановая risk capacity,Threshold,Фактическая доля risk decision,Количество risk decision
0,Narrow,10.0%,0.294522,10.00%,28962
1,Moderate,15.0%,0.182738,15.00%,43442
2,Balanced,20.0%,0.123821,20.00%,57923
3,Broad,25.0%,0.088776,25.00%,72404
4,High-recall,30.0%,0.066224,30.00%,86884



Количество режимов: 5
Метрики FP/FN рассчитаны: False
Final test использован: False


# 3. FP/FN-карта текущего baseline

Operating modes были зафиксированы до просмотра результатов ошибок.

Теперь для каждого режима считаем confusion matrix на одном и том же
сохранённом `GBDT_mean` OOF:

- `TP` — дефолт правильно отмечен как риск;
- `FN` — дефолт пропущен;
- `FP` — недефолт ошибочно отмечен как риск;
- `TN` — недефолт правильно оставлен вне risk decision.

Основные показатели:

- **Recall** показывает, какую долю дефолтов обнаруживает режим;
- **Precision** показывает, какая доля risk decisions действительно относится к дефолтам;
- **FNR** показывает долю пропущенных дефолтов;
- **FPR** показывает долю недефолтов, ошибочно попавших в risk decision.

Бизнес-reference `Recall ≈ 69%` используется только как ориентир.
Threshold специально под него не подбирается.

Все расчёты выполняются только на working OOF.
Final test не используется.

In [6]:
# ============================================================
# 3.1 Расчёт FP/FN для заранее зафиксированных operating modes
#
# Ничего не подбираем.
# Используем thresholds, определённые в разделе 2.
# ============================================================

fpfn_rows = []

for row in mode_rows:
    mode_name = row["Режим"]
    threshold = row["Threshold"]

    y_pred = (gbdt_mean_oof >= threshold).astype(int)

    tp = int(np.sum((y_oof == 1) & (y_pred == 1)))
    fn = int(np.sum((y_oof == 1) & (y_pred == 0)))
    fp = int(np.sum((y_oof == 0) & (y_pred == 1)))
    tn = int(np.sum((y_oof == 0) & (y_pred == 0)))

    assert tp + fn == int(np.sum(y_oof == 1))
    assert fp + tn == int(np.sum(y_oof == 0))
    assert tp + fn + fp + tn == len(y_oof)

    recall = tp / (tp + fn)
    precision = tp / (tp + fp)
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    fnr = fn / (tp + fn)
    fpr = fp / (fp + tn)

    positive_share = (tp + fp) / len(y_oof)

    fpfn_rows.append(
        {
            "Режим": mode_name,
            "Threshold": threshold,
            "Risk capacity": positive_share,
            "TP": tp,
            "FN": fn,
            "FP": fp,
            "TN": tn,
            "Recall": recall,
            "Precision": precision,
            "F1": f1,
            "FNR": fnr,
            "FPR": fpr,
            "Δ Recall до 69%": recall - 0.69,
        }
    )


fpfn_table = pd.DataFrame(fpfn_rows)


print("FP/FN-карта GBDT_mean OOF")
print("-------------------------")

display(
    fpfn_table.style.format(
        {
            "Threshold": "{:.6f}",
            "Risk capacity": "{:.1%}",
            "Recall": "{:.2%}",
            "Precision": "{:.2%}",
            "F1": "{:.3f}",
            "FNR": "{:.2%}",
            "FPR": "{:.2%}",
            "Δ Recall до 69%": "{:+.2%}",
        }
    )
)

print()
print("Контроль:")
print("working rows:", len(y_oof))
print("defaults:", int(y_oof.sum()))
print("final test used:", False)
print("model training:", False)
print("threshold optimization performed:", False)

FP/FN-карта GBDT_mean OOF
-------------------------


,Режим,Threshold,Risk capacity,TP,FN,FP,TN,Recall,Precision,F1,FNR,FPR,Δ Recall до 69%
0,Narrow,0.294522,10.0%,16086,11929,12876,248723,57.42%,55.54%,0.565,42.58%,4.92%,-11.58%
1,Moderate,0.182738,15.0%,19594,8421,23848,237751,69.94%,45.10%,0.548,30.06%,9.12%,+0.94%
2,Balanced,0.123821,20.0%,21864,6151,36059,225540,78.04%,37.75%,0.509,21.96%,13.78%,+9.04%
3,Broad,0.088776,25.0%,23465,4550,48939,212660,83.76%,32.41%,0.467,16.24%,18.71%,+14.76%
4,High-recall,0.066224,30.0%,24506,3509,62378,199221,87.47%,28.21%,0.427,12.53%,23.84%,+18.47%



Контроль:
working rows: 289614
defaults: 28015
final test used: False
model training: False
threshold optimization performed: False


# 4. Поведение 805 common blind-spot defaults

Общая группа blind spot была зафиксирована ранее в Stage 3:

- `DefMark = 1`;
- `consensus_rank <= 0.50`;
- `rank_spread <= 0.10`.

Размер принятой группы — **805 клиентов**.

Теперь проверяем, сколько этих 805 случаев попадает в положительный
risk decision при каждом из заранее зафиксированных operating modes.

Это важно разделить от общего Recall.

Высокий Recall на всей выборке ещё не означает,
что режим решает именно проблему common blind spot.

Новые модели не обучаются.
Определение blind spot не меняется.
Final test не используется.

In [7]:
# ============================================================
# 4.1 Проверка 805 common blind spot в operating modes
#
# Используем точное принятое правило Stage 3.
# Ничего не переопределяем и не подбираем.
# ============================================================

common_blind_mask = (
    (y_oof == 1)
    & (consensus_rank <= 0.50)
    & (rank_spread <= 0.10)
)

common_blind_count = int(common_blind_mask.sum())

assert common_blind_count == 805, (
    f"Ожидалось 805 common blind defaults, "
    f"получено {common_blind_count}"
)


blind_rows = []

for row in mode_rows:
    mode_name = row["Режим"]
    threshold = row["Threshold"]

    y_pred = (gbdt_mean_oof >= threshold).astype(int)

    captured = int(
        np.sum(common_blind_mask & (y_pred == 1))
    )

    missed = common_blind_count - captured

    capture_rate = captured / common_blind_count

    blind_rows.append(
        {
            "Режим": mode_name,
            "Threshold": threshold,
            "Blind spot captured": captured,
            "Blind spot missed": missed,
            "Blind spot capture rate": capture_rate,
        }
    )


blind_operating_table = pd.DataFrame(blind_rows)


print("805 common blind spot по operating modes")
print("----------------------------------------")

display(
    blind_operating_table.style.format(
        {
            "Threshold": "{:.6f}",
            "Blind spot capture rate": "{:.2%}",
        }
    )
)

print()
print("Контроль:")
print("common blind spot:", common_blind_count)
print("final test used:", False)
print("model training:", False)
print("blind spot definition changed:", False)

805 common blind spot по operating modes
----------------------------------------


,Режим,Threshold,Blind spot captured,Blind spot missed,Blind spot capture rate
0,Narrow,0.294522,0,805,0.00%
1,Moderate,0.182738,0,805,0.00%
2,Balanced,0.123821,0,805,0.00%
3,Broad,0.088776,0,805,0.00%
4,High-recall,0.066224,0,805,0.00%



Контроль:
common blind spot: 805
final test used: False
model training: False
blind spot definition changed: False


## 4.2 Насколько глубоко нужно расширить risk zone для захвата blind spot

В operating modes с risk capacity от 10% до 30%
не был захвачен ни один из 805 common blind-spot defaults.

При этом общий Recall baseline вырос с 57.42% до 87.47%.

Это показывает, что повышение общего Recall само по себе
не означает улучшения именно на common blind spot.

Однако результат требует осторожной трактовки:

группа 805 была изначально определена через низкий `consensus_rank`,
поэтому её положение в нижней части общего risk ranking
частично связано с самим определением blind spot.

Следующий диагностический вопрос:

**какую долю всей working sample пришлось бы отправить в risk decision,
чтобы начать захватывать эту группу и затем поймать 10%, 25%, 50% и 75%
common blind spot?**

Это не поиск оптимального threshold.

Мы только измеряем операционную цену
попытки решить blind spot снижением порога существующего baseline.

In [8]:
# ============================================================
# 4.2 Операционная цена захвата common blind spot
#
# Для каждого клиента определяем его положение
# в ranking GBDT_mean OOF.
#
# Затем смотрим, какую минимальную risk capacity
# пришлось бы открыть для захвата заданной доли 805 blind spot.
#
# Это diagnostic analysis, а не threshold optimization.
# ============================================================

n = len(gbdt_mean_oof)

# 1 = самый высокий риск.
order = np.argsort(-gbdt_mean_oof)

rank_position = np.empty(n, dtype=int)
rank_position[order] = np.arange(1, n + 1)

risk_capacity_position = rank_position / n

blind_capacities = np.sort(
    risk_capacity_position[common_blind_mask]
)


capture_targets = [
    ("Первый клиент", 1 / common_blind_count),
    ("1%", 0.01),
    ("10%", 0.10),
    ("25%", 0.25),
    ("50%", 0.50),
    ("75%", 0.75),
    ("100%", 1.00),
]


capacity_rows = []

for label, target_rate in capture_targets:
    target_count = max(
        1,
        int(np.ceil(common_blind_count * target_rate))
    )

    target_count = min(
        target_count,
        common_blind_count
    )

    required_capacity = float(
        blind_capacities[target_count - 1]
    )

    cutoff_position = int(
        np.ceil(required_capacity * n)
    )

    selected_mask = (
        risk_capacity_position <= required_capacity
    )

    captured_blind = int(
        np.sum(
            common_blind_mask
            & selected_mask
        )
    )

    fp_at_capacity = int(
        np.sum(
            (y_oof == 0)
            & selected_mask
        )
    )

    tp_at_capacity = int(
        np.sum(
            (y_oof == 1)
            & selected_mask
        )
    )

    capacity_rows.append(
        {
            "Цель захвата blind spot": label,
            "Минимум blind клиентов": target_count,
            "Требуемая risk capacity": required_capacity,
            "Всего risk decisions": int(selected_mask.sum()),
            "Blind captured": captured_blind,
            "TP всего": tp_at_capacity,
            "FP всего": fp_at_capacity,
        }
    )


blind_capacity_table = pd.DataFrame(
    capacity_rows
)


print("Операционная цена захвата common blind spot")
print("-------------------------------------------")

display(
    blind_capacity_table.style.format(
        {
            "Требуемая risk capacity": "{:.2%}",
        }
    )
)

print()
print("Контроль:")
print("common blind spot:", common_blind_count)
print(
    "минимальная capacity для первого blind case:",
    f"{blind_capacities[0]:.2%}",
)
print("final test used:", False)
print("model training:", False)
print("threshold optimization performed:", False)

Операционная цена захвата common blind spot
-------------------------------------------


,Цель захвата blind spot,Минимум blind клиентов,Требуемая risk capacity,Всего risk decisions,Blind captured,TP всего,FP всего
0,Первый клиент,1,50.11%,145114,1,26755,118359
1,1%,9,50.32%,145732,9,26772,118960
2,10%,81,52.10%,150881,81,26881,124000
3,25%,202,55.78%,161543,202,27071,134472
4,50%,403,63.51%,183938,403,27392,156546
5,75%,604,76.34%,221091,604,27729,193362
6,100%,805,98.40%,284977,805,28015,256962



Контроль:
common blind spot: 805
минимальная capacity для первого blind case: 50.11%
final test used: False
model training: False
threshold optimization performed: False


## 4.3 Вывод: можно ли решить blind spot только снижением threshold

### Факты

Ни один из 805 common blind-spot defaults не попадает
в заранее зафиксированные operating modes с risk capacity от 10% до 30%.

Чтобы существующий `GBDT_mean` начал захватывать эту группу только за счёт
расширения общей risk zone, потребовалось бы:

- **50.11%** выборки — чтобы захватить первый случай;
- **52.10%** — чтобы захватить 10% blind spot;
- **63.51%** — чтобы захватить 50%;
- **76.34%** — чтобы захватить 75%;
- **98.40%** — чтобы захватить все 805 случаев.

При захвате 50% blind spot в risk decision попало бы:

- **183 938** клиентов;
- из них **156 546** — недефолты (`FP`).

### Интерпретация

Для common blind spot простое снижение threshold
не выглядит практически эффективным механизмом.

Общий Recall можно существенно повысить снижением порога,
но это не решает специфическую проблему 805 common blind-spot defaults
без очень большого расширения risk zone и нагрузки по недефолтам.

Это важное различие:

**повысить общий Recall ≠ устранить common blind spot.**

### Ограничения

Группа 805 была изначально определена через низкий `consensus_rank`,
поэтому её низкое положение в общем risk ranking
частично связано с самим определением blind spot.

Результат не доказывает, что этих клиентов невозможно обнаружить
другими правилами, признаками или дополнительной информацией.

Он показывает только, что **изменение threshold текущего baseline**
не является эффективным решением этой конкретной группы.

### Следующий вопрос

Можно ли использовать уже существующий сигнал
**разногласия между моделями** как ограниченную зону дополнительной проверки
для части False Negative, не расширяя risk decision на половину выборки?

# 5. Может ли model disagreement использоваться как review signal

В предыдущем анализе простое снижение threshold оказалось неэффективным
для common blind spot.

Теперь проверяется другой уже существующий сигнал:
**разногласие между тремя baseline-моделями**.

Используется ранее зафиксированное правило Stage 3:

`rank_spread >= 0.25`

Новый порог disagreement не подбирается.

В качестве reference operating mode используется `Moderate`,
поскольку среди заранее зафиксированных режимов он оказался
ближайшим к существующему бизнес-reference `Recall ≈ 69%`.

Это не означает, что `Moderate` признан оптимальным.

Исследовательский вопрос:

**содержит ли область model disagreement среди клиентов,
оставшихся вне risk decision, повышенную концентрацию False Negative
и может ли поэтому рассматриваться как кандидат на дополнительную проверку?**

Важно:

попадание клиента в review zone не означает,
что ручная проверка автоматически исправит решение.

Мы измеряем только потенциально полезную маршрутизацию ошибок.

In [9]:
# ============================================================
# 5.1 Проверка model disagreement как кандидата review zone
#
# Reference:
# Moderate operating mode — заранее зафиксированный режим,
# ближайший к business Recall reference ~69%.
#
# Review signal:
# rank_spread >= 0.25 — принятое правило Stage 3.
#
# Никакие новые thresholds не подбираются.
# ============================================================

MODERATE_MODE = next(
    row for row in mode_rows
    if row["Режим"] == "Moderate"
)

moderate_threshold = MODERATE_MODE["Threshold"]

moderate_pred = (
    gbdt_mean_oof >= moderate_threshold
).astype(int)

negative_decision_mask = (
    moderate_pred == 0
)

fn_mask = (
    (y_oof == 1)
    & negative_decision_mask
)

disagreement_mask = (
    rank_spread >= 0.25
)

review_candidate_mask = (
    negative_decision_mask
    & disagreement_mask
)


negative_count = int(
    negative_decision_mask.sum()
)

fn_count = int(
    fn_mask.sum()
)

review_count = int(
    review_candidate_mask.sum()
)

review_defaults = int(
    np.sum(
        review_candidate_mask
        & (y_oof == 1)
    )
)

review_nondefaults = int(
    np.sum(
        review_candidate_mask
        & (y_oof == 0)
    )
)

fn_routed_to_review = int(
    np.sum(
        review_candidate_mask
        & fn_mask
    )
)

blind_routed_to_review = int(
    np.sum(
        review_candidate_mask
        & common_blind_mask
    )
)


negative_default_rate = (
    fn_count / negative_count
)

review_default_rate = (
    review_defaults / review_count
    if review_count > 0
    else 0.0
)

fn_review_share = (
    fn_routed_to_review / fn_count
    if fn_count > 0
    else 0.0
)

review_load_total = (
    review_count / len(y_oof)
)

review_load_negative = (
    review_count / negative_count
)

enrichment = (
    review_default_rate / negative_default_rate
    if negative_default_rate > 0
    else np.nan
)


review_disagreement_table = pd.DataFrame(
    [
        {
            "Reference mode": "Moderate",
            "Recall reference mode": (
                np.sum(
                    (y_oof == 1)
                    & (moderate_pred == 1)
                )
                / np.sum(y_oof == 1)
            ),
            "Negative decisions": negative_count,
            "FN всего": fn_count,
            "Review candidates": review_count,
            "Review load от всей выборки": review_load_total,
            "Review load среди negative decisions": review_load_negative,
            "FN routed to review": fn_routed_to_review,
            "Доля FN routed to review": fn_review_share,
            "Недефолты в review": review_nondefaults,
            "Default rate среди negative decisions": negative_default_rate,
            "Default rate в review zone": review_default_rate,
            "Default enrichment": enrichment,
            "Common blind routed": blind_routed_to_review,
        }
    ]
)


print("Model disagreement как кандидат review zone")
print("-------------------------------------------")

display(
    review_disagreement_table.style.format(
        {
            "Recall reference mode": "{:.2%}",
            "Review load от всей выборки": "{:.2%}",
            "Review load среди negative decisions": "{:.2%}",
            "Доля FN routed to review": "{:.2%}",
            "Default rate среди negative decisions": "{:.2%}",
            "Default rate в review zone": "{:.2%}",
            "Default enrichment": "{:.2f}x",
        }
    )
)

print()
print("Контроль:")
print("disagreement rule: rank_spread >= 0.25")
print("new disagreement threshold selected:", False)
print("final test used:", False)
print("model training:", False)

Model disagreement как кандидат review zone
-------------------------------------------


,Reference mode,Recall reference mode,Negative decisions,FN всего,Review candidates,Review load от всей выборки,Review load среди negative decisions,FN routed to review,Доля FN routed to review,Недефолты в review,Default rate среди negative decisions,Default rate в review zone,Default enrichment,Common blind routed
0,Moderate,69.94%,246172,8421,1762,0.61%,0.72%,32,0.38%,1730,3.42%,1.82%,0.53x,0



Контроль:
disagreement rule: rank_spread >= 0.25
new disagreement threshold selected: False
final test used: False
model training: False


## 5.2 Вывод по model disagreement

### Факты

Для reference-режима `Moderate`:

- общий Recall = **69.94%**;
- вне risk decision остаётся **8 421 FN**;
- правило Stage 3 `rank_spread >= 0.25` выделяет только **1 762 клиента**;
- это **0.61%** всей working sample;
- в эту review zone попадает только **32 FN**;
- это **0.38%** всех FN режима `Moderate`;
- default rate среди всех negative decisions = **3.42%**;
- default rate внутри disagreement zone = **1.82%**;
- enrichment = **0.53x**.

То есть disagreement zone не концентрирует пропущенные дефолты.
Наоборот, доля дефолтов внутри неё ниже, чем среди negative decisions в целом.

Для 805 common blind-spot defaults:

- routed to review = **0**.

Но это значение нельзя трактовать как независимый результат,
поскольку common blind spot определяется условием `rank_spread <= 0.10`,
а disagreement zone — условием `rank_spread >= 0.25`.
Эти определения взаимно исключают друг друга.

### Интерпретация

На текущем evidence model disagreement
**не является полезным кандидатом на дополнительную проверку**
для снижения False Negative в режиме `Moderate`.

Небольшая review load сама по себе недостаточна:
зона должна также концентрировать ошибки,
а здесь этого не происходит.

### Ограничения

Результат относится именно к ранее принятому правилу
`rank_spread >= 0.25`.

Он не доказывает, что любые формы model uncertainty
или disagreement бесполезны вообще.

Новый disagreement threshold в Stage 18 не подбирается.

### Следующий вопрос

Можно ли сформировать практическую review zone
не через disagreement моделей,
а через **близость score к границе принятия решения**?

Идея проста:

клиенты далеко выше threshold имеют достаточно сильный risk signal,
клиенты далеко ниже threshold — слабый.

Наиболее естественным кандидатом для дополнительной проверки
является ограниченная зона непосредственно вокруг operating threshold.

# 6. Зона дополнительной проверки вокруг operating threshold

Model disagreement не показал полезной концентрации False Negative.

Следующая гипотеза основана не на новой модели,
а на положении клиента относительно границы решения.

Для reference-режима `Moderate` threshold отделяет верхние 15%
клиентов по OOF risk score.

Клиенты непосредственно около этой границы
являются естественными кандидатами на дополнительную проверку,
поскольку небольшое изменение score меняет их operating decision.

До просмотра результата фиксируются три размера review zone:

- **2%** всей working sample;
- **5%**;
- **10%**.

Каждая зона располагается симметрично вокруг границы
`Moderate` в risk ranking.

Никакая ширина не выбирается по Recall, Precision, FP или FN.

Цель анализа — не найти «лучший» размер,
а показать бизнесу компромисс:

**review load ↔ количество решений около границы ↔ количество FP/FN,
которые попадают в дополнительную проверку.**

In [10]:
# ============================================================
# 6.1 Review zone вокруг Moderate boundary
#
# Фиксированные размеры:
# 2%, 5%, 10% working sample.
#
# Зона строится по risk ranking, а не подбирается по ошибкам.
# ============================================================

MODERATE_CAPACITY = 0.15

REVIEW_ZONE_SIZES = [
    0.02,
    0.05,
    0.10,
]


# percentile позиции:
# 0 = самый высокий риск,
# 1 = самый низкий риск
risk_order = np.argsort(-gbdt_mean_oof)

risk_fraction = np.empty(len(gbdt_mean_oof), dtype=float)
risk_fraction[risk_order] = (
    np.arange(len(gbdt_mean_oof)) + 0.5
) / len(gbdt_mean_oof)


boundary_rows = []

for zone_size in REVIEW_ZONE_SIZES:
    half_width = zone_size / 2

    lower = MODERATE_CAPACITY - half_width
    upper = MODERATE_CAPACITY + half_width

    review_mask = (
        (risk_fraction >= lower)
        & (risk_fraction < upper)
    )

    review_count = int(review_mask.sum())

    defaults_in_review = int(
        np.sum(
            review_mask
            & (y_oof == 1)
        )
    )

    nondefaults_in_review = int(
        np.sum(
            review_mask
            & (y_oof == 0)
        )
    )

    # Ошибки относительно Moderate decision:
    fp_in_review = int(
        np.sum(
            review_mask
            & (y_oof == 0)
            & (moderate_pred == 1)
        )
    )

    fn_in_review = int(
        np.sum(
            review_mask
            & (y_oof == 1)
            & (moderate_pred == 0)
        )
    )

    all_errors_in_review = (
        fp_in_review + fn_in_review
    )

    error_rate_in_review = (
        all_errors_in_review / review_count
        if review_count > 0
        else 0.0
    )

    fn_share_routed = (
        fn_in_review / fn_count
        if fn_count > 0
        else 0.0
    )

    fp_total_moderate = int(
        np.sum(
            (y_oof == 0)
            & (moderate_pred == 1)
        )
    )

    fp_share_routed = (
        fp_in_review / fp_total_moderate
        if fp_total_moderate > 0
        else 0.0
    )

    boundary_rows.append(
        {
            "Review zone": f"{zone_size:.0%}",
            "Нижняя граница ranking": lower,
            "Верхняя граница ranking": upper,
            "Review clients": review_count,
            "Defaults in review": defaults_in_review,
            "Non-defaults in review": nondefaults_in_review,
            "FN in review": fn_in_review,
            "Доля всех FN": fn_share_routed,
            "FP in review": fp_in_review,
            "Доля всех FP": fp_share_routed,
            "Всего ошибок в review": all_errors_in_review,
            "Error rate в review": error_rate_in_review,
        }
    )


boundary_review_table = pd.DataFrame(
    boundary_rows
)


print("Review zone вокруг Moderate threshold")
print("-------------------------------------")

display(
    boundary_review_table.style.format(
        {
            "Нижняя граница ranking": "{:.1%}",
            "Верхняя граница ranking": "{:.1%}",
            "Доля всех FN": "{:.2%}",
            "Доля всех FP": "{:.2%}",
            "Error rate в review": "{:.2%}",
        }
    )
)

print()
print("Контроль:")
print("reference mode:", "Moderate")
print("reference capacity:", "15%")
print("review zone sizes:", "2%, 5%, 10%")
print("zone widths selected from metrics:", False)
print("final test used:", False)
print("model training:", False)

Review zone вокруг Moderate threshold
-------------------------------------


,Review zone,Нижняя граница ranking,Верхняя граница ranking,Review clients,Defaults in review,Non-defaults in review,FN in review,Доля всех FN,FP in review,Доля всех FP,Всего ошибок в review,Error rate в review
0,2%,14.0%,16.0%,5792,1103,4689,548,6.51%,2341,9.82%,2889,49.88%
1,5%,12.5%,17.5%,14480,2811,11669,1264,15.01%,5693,23.87%,6957,48.05%
2,10%,10.0%,20.0%,28962,5779,23183,2270,26.96%,10972,46.01%,13242,45.72%



Контроль:
reference mode: Moderate
reference capacity: 15%
review zone sizes: 2%, 5%, 10%
zone widths selected from metrics: False
final test used: False
model training: False


## 6.2 Вывод по review zone около threshold

### Факты

Для reference-режима `Moderate` (`Recall = 69.94%`) были заранее
зафиксированы три размера зоны дополнительной проверки.

При review zone **2%**:

- на дополнительную проверку попадает **5 792 клиента**;
- внутри зоны находится **548 FN**;
- это **6.51% всех FN** режима `Moderate`;
- также в зоне находится **2 341 FP**;
- это **9.82% всех FP**;
- суммарная доля ошибочных решений внутри review zone составляет **49.88%**.

При review zone **5%**:

- review load = **14 480 клиентов**;
- routed FN = **1 264** (**15.01% всех FN**);
- routed FP = **5 693** (**23.87% всех FP**);
- error rate внутри зоны = **48.05%**.

При review zone **10%**:

- review load = **28 962 клиента**;
- routed FN = **2 270** (**26.96% всех FN**);
- routed FP = **10 972** (**46.01% всех FP**);
- error rate внутри зоны = **45.72%**.

### Интерпретация

В отличие от model disagreement,
область непосредственно около operating threshold
действительно концентрирует ошибки бинарного решения.

Это делает score-boundary естественным кандидатом
для **зоны дополнительной проверки**.

Практически появляется трёхзонный процесс:

`высокий риск → дополнительная проверка → обычный поток`

вместо попытки одним threshold автоматически решить все случаи.

При этом Stage 18 пока **не выбирает оптимальную ширину review zone**.

Размер 2%, 5% или 10% должен определяться уже допустимой
операционной нагрузкой и стоимостью FN/FP.

### Важное ограничение

Попадание ошибки в review zone не означает,
что ручная или бизнес-проверка обязательно её исправит.

Полученный результат показывает только,
что review process будет направлен в область,
где ошибки текущего бинарного решения существенно концентрируются.

Также этот механизм не решает common blind spot из 805 клиентов:
они расположены значительно глубже в low-risk ranking
и требуют отдельного информационного механизма.

### Следующий вопрос

Насколько сильнее ошибки концентрируются внутри review zone
по сравнению со всей working sample и с клиентами вне этой зоны?

Это позволит проверить,
действительно ли дополнительная проверка направляется
в более информативную область, а не просто увеличивает workload.

In [11]:
# ============================================================
# 7.1 Концентрация ошибок внутри threshold review zone
#
# Сравниваем:
# - общий error rate Moderate;
# - error rate внутри review zone;
# - error rate вне review zone.
#
# Размеры зон уже зафиксированы ранее.
# Нового выбора по результату нет.
# ============================================================

moderate_error_mask = (
    ((y_oof == 1) & (moderate_pred == 0))
    | ((y_oof == 0) & (moderate_pred == 1))
)

overall_error_rate = float(
    moderate_error_mask.mean()
)

concentration_rows = []

for zone_size in REVIEW_ZONE_SIZES:
    half_width = zone_size / 2

    lower = MODERATE_CAPACITY - half_width
    upper = MODERATE_CAPACITY + half_width

    review_mask = (
        (risk_fraction >= lower)
        & (risk_fraction < upper)
    )

    outside_mask = ~review_mask

    review_error_rate = float(
        moderate_error_mask[review_mask].mean()
    )

    outside_error_rate = float(
        moderate_error_mask[outside_mask].mean()
    )

    enrichment_vs_all = (
        review_error_rate / overall_error_rate
    )

    enrichment_vs_outside = (
        review_error_rate / outside_error_rate
    )

    concentration_rows.append(
        {
            "Review zone": f"{zone_size:.0%}",
            "Review clients": int(review_mask.sum()),
            "Overall error rate": overall_error_rate,
            "Error rate в review": review_error_rate,
            "Error rate вне review": outside_error_rate,
            "Enrichment vs all": enrichment_vs_all,
            "Enrichment vs outside": enrichment_vs_outside,
        }
    )


error_concentration_table = pd.DataFrame(
    concentration_rows
)


print("Концентрация ошибок около Moderate threshold")
print("--------------------------------------------")

display(
    error_concentration_table.style.format(
        {
            "Overall error rate": "{:.2%}",
            "Error rate в review": "{:.2%}",
            "Error rate вне review": "{:.2%}",
            "Enrichment vs all": "{:.2f}x",
            "Enrichment vs outside": "{:.2f}x",
        }
    )
)

print()
print("Контроль:")
print("reference mode:", "Moderate")
print("review zone widths optimized:", False)
print("final test used:", False)
print("model training:", False)

Концентрация ошибок около Moderate threshold
--------------------------------------------


,Review zone,Review clients,Overall error rate,Error rate в review,Error rate вне review,Enrichment vs all,Enrichment vs outside
0,2%,5792,11.14%,49.88%,10.35%,4.48x,4.82x
1,5%,14480,11.14%,48.05%,9.20%,4.31x,5.22x
2,10%,28962,11.14%,45.72%,7.30%,4.10x,6.26x



Контроль:
reference mode: Moderate
review zone widths optimized: False
final test used: False
model training: False


## 7.2 Практический смысл концентрации ошибок

### Факты

Для reference-режима `Moderate` общий error rate бинарного решения составляет **11.14%**.

При этом около operating threshold:

- review zone **2%** имеет error rate **49.88%**;
- review zone **5%** — **48.05%**;
- review zone **10%** — **45.72%**.

Концентрация ошибок относительно всей working sample составляет:

- **4.48x** для зоны 2%;
- **4.31x** для зоны 5%;
- **4.10x** для зоны 10%.

Относительно клиентов вне review zone концентрация ещё выше:

- **4.82x**;
- **5.22x**;
- **6.26x** соответственно.

### Интерпретация

Граница operating decision является не случайной областью,
а зоной повышенной неопределённости бинарного решения.

Поэтому review zone вокруг threshold имеет практический смысл:
она позволяет направить дополнительную проверку туда,
где вероятность ошибки заметно выше средней.

Это отличается от простого снижения threshold:

- снижение threshold увеличивает общий Recall,
  но одновременно расширяет risk decision на большое число недефолтов;
- review zone сохраняет основной operating mode,
  но выделяет ограниченную область повышенной концентрации ошибок.

### Ограничения

Stage 18 не доказывает,
что дополнительная проверка сможет исправить каждую ошибку в review zone.

Также без стоимости ручной проверки, FN и FP
нельзя выбрать оптимальный размер зоны.

Поэтому 2%, 5% и 10% остаются **operating scenarios**,
а не оптимизированными решениями.

In [12]:
# ============================================================
# 7.3 Workload vs доля перехваченных ошибок
#
# Проверяем:
# какую долю всех ошибок Moderate можно направить
# на дополнительную проверку при review load 2%, 5%, 10%.
#
# Размеры зон уже зафиксированы.
# Никакой оптимизации здесь нет.
# ============================================================

total_moderate_errors = int(
    moderate_error_mask.sum()
)

workload_rows = []

for row in boundary_rows:
    review_clients = int(row["Review clients"])
    errors_in_review = int(row["Всего ошибок в review"])

    review_load = (
        review_clients / len(y_oof)
    )

    error_capture_share = (
        errors_in_review / total_moderate_errors
    )

    capture_efficiency = (
        error_capture_share / review_load
        if review_load > 0
        else np.nan
    )

    workload_rows.append(
        {
            "Review zone": row["Review zone"],
            "Review clients": review_clients,
            "Review load": review_load,
            "Ошибок направлено в review": errors_in_review,
            "Доля всех ошибок": error_capture_share,
            "Error capture / workload": capture_efficiency,
        }
    )


review_workload_table = pd.DataFrame(
    workload_rows
)


print("Workload vs перехваченные ошибки")
print("--------------------------------")

display(
    review_workload_table.style.format(
        {
            "Review load": "{:.2%}",
            "Доля всех ошибок": "{:.2%}",
            "Error capture / workload": "{:.2f}x",
        }
    )
)

print()
print("Контроль:")
print("Всего ошибок Moderate:", total_moderate_errors)
print("Размеры review zone оптимизированы:", False)
print("Final test использован:", False)
print("Model training:", False)

Workload vs перехваченные ошибки
--------------------------------


,Review zone,Review clients,Review load,Ошибок направлено в review,Доля всех ошибок,Error capture / workload
0,2%,5792,2.00%,2889,8.95%,4.48x
1,5%,14480,5.00%,6957,21.56%,4.31x
2,10%,28962,10.00%,13242,41.04%,4.10x



Контроль:
Всего ошибок Moderate: 32269
Размеры review zone оптимизированы: False
Final test использован: False
Model training: False


# 8. Итог Stage 18 — FP/FN и operating modes

## Исследовательский вопрос

**Как меняется баланс False Positive и False Negative текущего baseline
при разных operating modes и можно ли обоснованно выделить
кандидата на зону дополнительной проверки?**

---

## ФАКТЫ

### 1. Threshold действительно управляет бизнес-компромиссом FP/FN

Для заранее зафиксированных operating modes `GBDT_mean`:

| Risk capacity | Recall | Precision | FN | FP |
|---:|---:|---:|---:|---:|
| 10% | 57.42% | 55.54% | 11 929 | 12 876 |
| 15% | 69.94% | 45.10% | 8 421 | 23 848 |
| 20% | 78.04% | 37.75% | 6 151 | 36 059 |
| 25% | 83.76% | 32.41% | 4 550 | 48 939 |
| 30% | 87.47% | 28.21% | 3 509 | 62 378 |

Расширение risk zone повышает Recall,
но одновременно существенно увеличивает количество False Positive.

Режим `Moderate` с capacity 15% дал Recall **69.94%**,
то есть оказался близок к существующему бизнес-reference около 69%.

Это не делает его математически или бизнес-оптимальным threshold.

---

### 2. Простое снижение threshold не решает common blind spot

Во всех заранее заданных operating modes от 10% до 30%
захвачено:

**0 из 805 common blind-spot defaults.**

Чтобы `GBDT_mean` начал захватывать эту группу
только за счёт расширения общей risk zone, требуется примерно:

- 50.11% sample — первый blind case;
- 52.10% — 10% blind spot;
- 63.51% — 50% blind spot;
- 76.34% — 75% blind spot;
- 98.40% — 100% blind spot.

Например, для захвата 50% группы пришлось бы направить
в risk decision **183 938 клиентов**, включая **156 546 недефолтов**.

Следовательно:

**повышение общего Recall и решение common blind spot —
это разные задачи.**

---

### 3. Model disagreement не дал полезной review zone

Для режима `Moderate` правило Stage 3:

`rank_spread >= 0.25`

выделило 1 762 клиента, но туда попало только:

- **32 FN**;
- **0.38% всех FN**;
- default rate = **1.82%**

при default rate **3.42%** среди всех negative decisions.

Enrichment = **0.53x**.

Следовательно, принятое правило model disagreement
не концентрирует False Negative и не поддерживается evidence
как practical review signal.

`0` common blind cases в этой зоне не является независимым открытием:
определение common blind spot использует `rank_spread <= 0.10`,
поэтому две группы конструктивно не пересекаются.

---

### 4. Область около threshold действительно концентрирует ошибки

Для режима `Moderate` общий error rate бинарного решения:

**11.14%**.

При этом:

| Review load | Error rate внутри зоны | Доля всех ошибок, попавших в review |
|---:|---:|---:|
| 2% | 49.88% | 8.95% |
| 5% | 48.05% | 21.56% |
| 10% | 45.72% | 41.04% |

То есть:

- **2%** ручной зоны концентрируют **8.95% всех ошибок**;
- **5%** — **21.56% ошибок**;
- **10%** — **41.04% ошибок**.

Error concentration относительно всей sample составляет примерно:

- **4.48x**;
- **4.31x**;
- **4.10x**.

Таким образом, небольшая область около operating threshold
действительно содержит существенно более высокую концентрацию
ошибочных бинарных решений.

---

## ИНТЕРПРЕТАЦИЯ

Stage 18 показывает, что проблему ошибок нельзя свести
к поиску одного «идеального threshold».

Рациональнее разделить процесс на три уровня.

### 1. Operating threshold

Threshold задаёт общий бизнес-компромисс:

`Recall ↔ Precision ↔ FN ↔ FP ↔ объём risk decisions`.

Его окончательный выбор должен учитывать стоимость ошибок
и допустимую операционную нагрузку.

### 2. Пограничная review zone

Ограниченная область вокруг threshold является
обоснованным кандидатом на дополнительную проверку,
поскольку именно там ошибки бинарного решения
концентрируются значительно сильнее среднего.

Практический процесс может выглядеть как:

**явный высокий риск → дополнительная проверка → обычный поток**

вместо попытки заставить один threshold автоматически
решить все случаи.

### 3. Common blind spot

805 common blind-spot defaults находятся значительно глубже
в low-risk ranking.

Их проблему нельзя практически решить только:

- снижением threshold;
- расширением обычной risk zone;
- принятым правилом model disagreement.

Для этой группы остаётся отдельная гипотеза
о необходимости другого информационного сигнала
или иного механизма выявления.

---

## ОГРАНИЧЕНИЯ

Stage 18 не доказывает:

- какой threshold является бизнес-оптимальным;
- какой размер review zone необходимо внедрить;
- что ручная проверка исправит все routed FP/FN;
- что стоимость FN выше или ниже стоимости FP;
- что новый признак обязательно решит common blind spot;
- causal nature наблюдаемых ошибок.

Common blind spot был определён через model ranks,
поэтому его положение в low-risk ranking
частично связано с самим определением группы.

Random CV/OOF не доказывает temporal stability.

Final test не использовался.

Новые модели не обучались.

Новые признаки не создавались.

---

## РЕШЕНИЕ

Stage 18 поддерживает следующую схему будущего процесса Комуса:

**ML score → operating mode → пограничная review zone → анализ FP/FN → решение бизнеса.**

Threshold рассматривается как **business operating parameter**,
а не как способ скрыто оптимизировать модель.

Review zone вокруг threshold является
**кандидатом на дополнительную проверку**,
но её размер должен определяться бизнес-стоимостью и доступной capacity.

Common blind spot рассматривается отдельно
и не смешивается с обычными пограничными ошибками.

---

## Следующий шаг

Следующий research stage должен формализовать
**универсальный экспериментальный конвейер нового признака**:

`provenance`
→ `temporal admissibility`
→ `один controlled change`
→ `OOF quality`
→ `FP/FN`
→ `review zone`
→ `blind spot`
→ `ACCEPT / REJECT`.

Это превращает результаты проекта
из набора отдельных ML-экспериментов
в воспроизводимый исследовательский процесс Комуса.

In [13]:
# ============================================================
# 8.1 Сохранение evidence artifact Stage 18
#
# Сохраняем:
# - identity baseline / working OOF;
# - operating modes;
# - FP/FN map;
# - common blind spot diagnostics;
# - model disagreement check;
# - threshold review zones;
# - workload / error concentration;
# - итоговые ограничения и решение.
#
# Нового ML-run нет.
# Final test не используется.
# JSON должен быть strict-valid: NaN запрещён.
# ============================================================

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


ARTIFACT_PATH = (
    ROOT
    / "reports"
    / "generated"
    / "stage18_fp_fn_operating_modes_V1.json"
)


def to_jsonable(value):
    """
    Приводит pandas / numpy значения к strict JSON-compatible виду.
    NaN / inf -> None.
    """
    if value is None:
        return None

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        value = float(value)

    if isinstance(value, float):
        if not np.isfinite(value):
            return None
        return value

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, dict):
        return {
            str(key): to_jsonable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_jsonable(item)
            for item in value
        ]

    return value


def table_records(df):
    """
    DataFrame -> strict JSON-compatible records.
    """
    clean = (
        df.astype(object)
        .where(pd.notna(df), None)
        .to_dict(orient="records")
    )

    return to_jsonable(clean)


moderate_row = fpfn_table.loc[
    fpfn_table["Режим"] == "Moderate"
].iloc[0]


stage18_artifact = {
    "stage": "Stage 18",
    "version": "V1",

    "research_question": (
        "Как меняется баланс False Positive и False Negative "
        "текущего baseline при разных operating modes "
        "и можно ли обоснованно выделить кандидата "
        "на зону дополнительной проверки?"
    ),

    "status": "FP_FN_OPERATING_MAP_COMPLETE",

    "experiment_flags": {
        "model_training": False,
        "new_features_created": False,
        "final_test_used": False,
        "threshold_optimization_performed": False,
        "review_zone_widths_optimized": False,
        "new_disagreement_threshold_selected": False,
    },

    "data_identity": {
        "working_rows": int(len(y_oof)),
        "defaults": int(y_oof.sum()),
        "non_defaults": int((y_oof == 0).sum()),
        "common_blind_spot": int(common_blind_count),
    },

    "baseline_identity": {
        "name": "GBDT_mean",
        "source": (
            "Mean of saved CatBoost, XGBoost and LightGBM "
            "working-sample OOF probabilities"
        ),
        "oof_gini_reconstructed": float(gini),
        "accepted_oof_gini_reference": float(EXPECTED_GINI),
        "gini_tolerance": float(GINI_TOLERANCE),
    },

    "operating_modes": table_records(
        operating_modes_table
    ),

    "fp_fn_map": table_records(
        fpfn_table
    ),

    "common_blind_spot_by_operating_mode": table_records(
        blind_operating_table
    ),

    "common_blind_spot_capacity_cost": table_records(
        blind_capacity_table
    ),

    "model_disagreement_review_candidate": table_records(
        review_disagreement_table
    ),

    "threshold_review_zones": table_records(
        boundary_review_table
    ),

    "error_concentration": table_records(
        error_concentration_table
    ),

    "review_workload": table_records(
        review_workload_table
    ),

    "key_results": {
        "moderate": {
            "risk_capacity": float(
                moderate_row["Risk capacity"]
            ),
            "threshold": float(
                moderate_row["Threshold"]
            ),
            "recall": float(
                moderate_row["Recall"]
            ),
            "precision": float(
                moderate_row["Precision"]
            ),
            "fn": int(
                moderate_row["FN"]
            ),
            "fp": int(
                moderate_row["FP"]
            ),
        },

        "blind_spot": {
            "captured_at_10_to_30_percent_capacity": 0,
            "first_case_required_capacity": float(
                blind_capacities[0]
            ),
            "fifty_percent_required_capacity": float(
                blind_capacity_table.loc[
                    blind_capacity_table[
                        "Цель захвата blind spot"
                    ] == "50%",
                    "Требуемая risk capacity",
                ].iloc[0]
            ),
        },

        "disagreement": {
            "rule": "rank_spread >= 0.25",
            "fn_routed_to_review": int(
                review_disagreement_table[
                    "FN routed to review"
                ].iloc[0]
            ),
            "fn_share_routed_to_review": float(
                review_disagreement_table[
                    "Доля FN routed to review"
                ].iloc[0]
            ),
            "default_enrichment": float(
                review_disagreement_table[
                    "Default enrichment"
                ].iloc[0]
            ),
        },

        "threshold_review_zone": {
            "2_percent_error_rate": float(
                error_concentration_table.loc[
                    error_concentration_table[
                        "Review zone"
                    ] == "2%",
                    "Error rate в review",
                ].iloc[0]
            ),
            "2_percent_error_capture": float(
                review_workload_table.loc[
                    review_workload_table[
                        "Review zone"
                    ] == "2%",
                    "Доля всех ошибок",
                ].iloc[0]
            ),
            "5_percent_error_capture": float(
                review_workload_table.loc[
                    review_workload_table[
                        "Review zone"
                    ] == "5%",
                    "Доля всех ошибок",
                ].iloc[0]
            ),
            "10_percent_error_capture": float(
                review_workload_table.loc[
                    review_workload_table[
                        "Review zone"
                    ] == "10%",
                    "Доля всех ошибок",
                ].iloc[0]
            ),
        },
    },

    "facts": [
        (
            "При risk capacity 10–30% общий Recall растёт "
            "с 57.42% до 87.47%, одновременно растёт FP-нагрузка."
        ),
        (
            "Moderate mode с capacity 15% дал Recall около 69.94%, "
            "но не признан оптимальным threshold."
        ),
        (
            "Ни один из 805 common blind-spot defaults "
            "не попал в operating modes 10–30%."
        ),
        (
            "Первый common blind case появляется только "
            "примерно после 50.11% общей risk capacity."
        ),
        (
            "Принятое правило model disagreement "
            "rank_spread >= 0.25 не концентрирует FN."
        ),
        (
            "Review zone вокруг Moderate threshold "
            "концентрирует ошибки значительно сильнее "
            "working sample в целом."
        ),
        (
            "При review load 2%, 5% и 10% "
            "в review попадает примерно 8.95%, 21.56% "
            "и 41.04% всех ошибок Moderate mode."
        ),
    ],

    "interpretation": (
        "FP/FN следует разделять на общий operating trade-off, "
        "пограничные ошибки около threshold и отдельный common blind spot. "
        "Threshold управляет общим Recall/FP balance; "
        "пограничная review zone является кандидатом "
        "на дополнительную проверку; common blind spot "
        "не решается практически одним снижением threshold."
    ),

    "limitations": [
        (
            "Stage 18 не определяет бизнес-оптимальный threshold "
            "без стоимости FN, FP и review workload."
        ),
        (
            "Попадание ошибки в review zone не означает, "
            "что дополнительная проверка обязательно исправит решение."
        ),
        (
            "Размеры review zone 2%, 5% и 10% "
            "не оптимизировались по метрикам."
        ),
        (
            "Common blind spot определён через model ranks, "
            "поэтому его низкое положение в ranking "
            "частично связано с определением группы."
        ),
        (
            "Model disagreement result относится к принятому "
            "правилу rank_spread >= 0.25 и не доказывает "
            "бесполезность любых uncertainty signals."
        ),
        (
            "Random CV/OOF не доказывает temporal stability."
        ),
    ],

    "decision": {
        "threshold": (
            "Рассматривать как business operating parameter, "
            "а не как способ скрытой оптимизации модели."
        ),
        "review_zone": (
            "Score-boundary zone поддерживается evidence "
            "как кандидат на дополнительную проверку; "
            "размер должен определяться business cost/capacity."
        ),
        "model_disagreement": (
            "Принятое правило rank_spread >= 0.25 "
            "не поддерживается как practical FN review signal."
        ),
        "common_blind_spot": (
            "Не смешивать с обычными пограничными ошибками; "
            "для него требуется отдельный информационный "
            "или иной механизм выявления."
        ),
        "next_research_stage": (
            "Формализация универсального controlled pipeline "
            "для проверки нового признака."
        ),
    },

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


stage18_artifact = to_jsonable(
    stage18_artifact
)


ARTIFACT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


with ARTIFACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        stage18_artifact,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )


print("✅ Артефакт Stage 18 сохранён:")
print(ARTIFACT_PATH.resolve())

print()
print("Контроль:")
print(
    "status:",
    stage18_artifact["status"],
)
print(
    "working rows:",
    stage18_artifact[
        "data_identity"
    ]["working_rows"],
)
print(
    "common blind:",
    stage18_artifact[
        "data_identity"
    ]["common_blind_spot"],
)
print(
    "Moderate Recall:",
    f"{stage18_artifact['key_results']['moderate']['recall']:.2%}",
)
print(
    "2% review -> errors:",
    f"{stage18_artifact['key_results']['threshold_review_zone']['2_percent_error_capture']:.2%}",
)
print(
    "5% review -> errors:",
    f"{stage18_artifact['key_results']['threshold_review_zone']['5_percent_error_capture']:.2%}",
)
print(
    "10% review -> errors:",
    f"{stage18_artifact['key_results']['threshold_review_zone']['10_percent_error_capture']:.2%}",
)
print(
    "final_test_used:",
    stage18_artifact[
        "experiment_flags"
    ]["final_test_used"],
)
print(
    "model_training:",
    stage18_artifact[
        "experiment_flags"
    ]["model_training"],
)

✅ Артефакт Stage 18 сохранён:
D:\Projects\komus-work\reports\generated\stage18_fp_fn_operating_modes_V1.json

Контроль:
status: FP_FN_OPERATING_MAP_COMPLETE
working rows: 289614
common blind: 805
Moderate Recall: 69.94%
2% review -> errors: 8.95%
5% review -> errors: 21.56%
10% review -> errors: 41.04%
final_test_used: False
model_training: False


In [14]:
# ============================================================
# 8.2 Provenance closeout Stage 18
#
# Добавляем в evidence artifact:
# - dataset identity;
# - working/final split identity;
# - exact 47-feature contract;
# - подтверждение исключения Q_B1_norm / Q_B2_norm.
#
# Никаких новых моделей, метрик или threshold search.
# ============================================================

STAGE17_PATH = (
    ROOT
    / "reports"
    / "generated"
    / "stage17_information_gap_map_V1.json"
)

assert STAGE17_PATH.exists(), (
    f"Не найден Stage 17 artifact: {STAGE17_PATH}"
)

with STAGE17_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    stage17_identity = json.load(file)


feature_contract = stage17_identity["feature_contract"]

assert feature_contract["allowed_feature_count"] == 47
assert feature_contract["Q_B1_norm_used_as_predictor"] is False
assert feature_contract["Q_B2_norm_used_as_predictor"] is False


with ARTIFACT_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    stage18_final = json.load(file)


stage18_final["dataset_contract"] = {
    "dataset": "Data_final.xlsb",
    "dataset_sha256": (
        "fc742be66d238c529daba52ccc755f774"
        "f836b7d052ed062cdf0b345080e7930"
    ),

    "target": "DefMark",
    "identifier": "INN",

    "working_rows": 289614,
    "final_test_rows": 72404,

    "working_index_sha256": (
        "80430ce6290d0982d3641621ba1ed62f"
        "6fb495e8d32f7d23d9fca00091aadb45"
    ),

    "final_test_index_sha256": (
        "35ec963fe82918841f4a8acb012e8fc8"
        "78aed66d2c5296ea6cb105177cafbb37"
    ),

    "split": {
        "working_share": 0.80,
        "final_test_share": 0.20,
        "seed": 42,
    },
}


stage18_final["feature_contract"] = {
    "allowed_feature_count": int(
        feature_contract["allowed_feature_count"]
    ),

    "allowed_features": (
        feature_contract["allowed_features"]
    ),

    "Q_B1_norm_used_as_predictor": False,
    "Q_B2_norm_used_as_predictor": False,
}


stage18_final["evidence_sources"] = {
    "oof_predictions": (
        "reports/generated/stage3_oof_predictions_V1.npz"
    ),
    "feature_contract_source": (
        "reports/generated/stage17_information_gap_map_V1.json"
    ),
}


with ARTIFACT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        stage18_final,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )


print("✅ Provenance Stage 18 добавлен")
print()
print("Контроль:")
print(
    "dataset:",
    stage18_final["dataset_contract"]["dataset"],
)
print(
    "working / final:",
    stage18_final["dataset_contract"]["working_rows"],
    "/",
    stage18_final["dataset_contract"]["final_test_rows"],
)
print(
    "allowed features:",
    stage18_final["feature_contract"]["allowed_feature_count"],
)
print(
    "Q_B1 predictor:",
    stage18_final["feature_contract"]["Q_B1_norm_used_as_predictor"],
)
print(
    "Q_B2 predictor:",
    stage18_final["feature_contract"]["Q_B2_norm_used_as_predictor"],
)
print(
    "final_test_used:",
    stage18_final["experiment_flags"]["final_test_used"],
)

✅ Provenance Stage 18 добавлен

Контроль:
dataset: Data_final.xlsb
working / final: 289614 / 72404
allowed features: 47
Q_B1 predictor: False
Q_B2 predictor: False
final_test_used: False
